In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import data
import contract

In [3]:
df = data.read_raw()

In [4]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
df.shape

(119390, 32)

In [6]:
df["arrival_date"] = data.arrival_date(df)

In [7]:
# Column-level data profiling

profile = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notnull().sum().values,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (df.isnull().mean() * 100).values,
    "Unique_Values": df.nunique(dropna=False).values
})

profile = profile.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(profile)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage,Unique_Values
24,company,float64,6797,112593,94.306893,353
23,agent,float64,103050,16340,13.686238,334
13,country,str,118902,488,0.408744,178
10,children,float64,119386,4,0.003350,6
0,hotel,str,119390,0,0.000000,2
5,arrival_date_week_number,int64,119390,0,0.000000,53
6,arrival_date_day_of_month,int64,119390,0,0.000000,31
1,is_canceled,int64,119390,0,0.000000,2
2,lead_time,int64,119390,0,0.000000,479
8,stays_in_week_nights,int64,119390,0,0.000000,35


In [8]:
df.shape

(119390, 33)

In [9]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
hotel,119390,2,City Hotel,79330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_canceled,119390.0,NaN,NaN,NaN,0.370416,0.0,0.0,0.0,1.0,1.0,0.482918
lead_time,119390.0,NaN,NaN,NaN,104.011416,0.0,18.0,69.0,160.0,737.0,106.863097
arrival_date_year,119390.0,NaN,NaN,NaN,2016.156554,2015.0,2016.0,2016.0,2017.0,2017.0,0.707476
arrival_date_month,119390,12,August,13877,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_date_week_number,119390.0,NaN,NaN,NaN,27.165173,1.0,16.0,28.0,38.0,53.0,13.605138
arrival_date_day_of_month,119390.0,NaN,NaN,NaN,15.798241,1.0,8.0,16.0,23.0,31.0,8.780829
stays_in_weekend_nights,119390.0,NaN,NaN,NaN,0.927599,0.0,0.0,1.0,2.0,19.0,0.998613
stays_in_week_nights,119390.0,NaN,NaN,NaN,2.500302,0.0,1.0,2.0,3.0,50.0,1.908286
adults,119390.0,NaN,NaN,NaN,1.856403,0.0,2.0,2.0,2.0,55.0,0.579261


In [10]:
df["market_segment"].value_counts()

market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64

### Auditing The client's claim

#### Checking The overall cancellation rate

In [12]:
df["is_canceled"].value_counts()

is_canceled
0    75166
1    44224
Name: count, dtype: int64

In [169]:
overall_cancellation = df["is_canceled"].mean()

print(
    f"Cancellation rate: "
    f"{overall_cancellation * 100:.2f}%"
)

Cancellation rate: 37.04%


In [170]:
hotel_cancellation = (
    df.groupby("hotel")["is_canceled"]
      .agg(["count", "mean"])
)
hotel_cancellation["mean"] =round(hotel_cancellation["mean"] *100, 2)

print(hotel_cancellation)

              count   mean
hotel                     
City Hotel    79330  41.73
Resort Hotel  40060  27.76


The cancellation rate differs significantly between the two hotels.

Therefore, using one overall cancellation rate for both hotels may result in poor booking decisions.

### Audit claim: Non-refundable bookings are generally guaranteed to become stays.

In [171]:
deposit_analysis = (
    df.groupby("deposit_type")
      .agg(
          bookings=("is_canceled", "count"),
          cancellation_rate=("is_canceled", "mean")
      )
)

deposit_analysis["cancellation_rate"] *= 100

print(deposit_analysis)

              bookings  cancellation_rate
deposit_type                             
No Deposit      104641          28.377022
Non Refund       14587          99.362446
Refundable         162          22.222222


In [172]:

# ==========================================
# STEP 2A: MISSING VALUES BY HOTEL
# ==========================================

# Identify columns containing missing values
missing_columns = df.columns[df.isnull().any()].tolist()

print("Columns containing missing values:")
print(missing_columns)


# Calculate missing values separately for each hotel
missing_by_hotel = (
    df.groupby("hotel")[missing_columns]
      .agg(lambda x: x.isnull().sum())
)

display(missing_by_hotel)


# Calculate missing percentage for each hotel
missing_percentage_by_hotel = (
    df.groupby("hotel")[missing_columns]
      .agg(lambda x: x.isnull().mean() * 100)
      .round(2)
)

print("Missing percentage by hotel:")
display(missing_percentage_by_hotel)


Columns containing missing values:
['children', 'country', 'agent', 'company']


,children,country,agent,company
hotel,,,,
City Hotel,4,24,8131,75641
Resort Hotel,0,464,8209,36952


Missing percentage by hotel:


,children,country,agent,company
hotel,,,,
City Hotel,0.01,0.03,10.25,95.35
Resort Hotel,0.00,1.16,20.49,92.24


In [173]:

# ==========================================
# STEP 2B: MISSING AGENT AND COMPANY ANALYSIS
# ==========================================

# Create indicators for missing agent and company
df["agent_missing"] = df["agent"].isnull()
df["company_missing"] = df["company"].isnull()


# Compare missingness against market segment
agent_analysis = pd.crosstab(
    df["market_segment"],
    df["agent_missing"],
    normalize="index"
) * 100

company_analysis = pd.crosstab(
    df["market_segment"],
    df["company_missing"],
    normalize="index"
) * 100


print("Agent missing percentage by market segment:")
display(agent_analysis.round(2))

print("Company missing percentage by market segment:")
display(company_analysis.round(2))


# Check booking volume for missing agent/company
print("Missing agent bookings:")
print(df["agent_missing"].sum())

print("\nMissing company bookings:")
print(df["company_missing"].sum())


Agent missing percentage by market segment:


agent_missing,False,True
market_segment,,
Aviation,10.55,89.45
Complementary,13.86,86.14
Corporate,13.24,86.76
Direct,52.35,47.65
Groups,79.06,20.94
Offline TA/TO,98.39,1.61
Online TA,99.38,0.62
Undefined,0.00,100.00


Company missing percentage by market segment:


company_missing,False,True
market_segment,,
Aviation,89.45,10.55
Complementary,42.13,57.87
Corporate,84.44,15.56
Direct,1.65,98.35
Groups,7.02,92.98
Offline TA/TO,0.44,99.56
Online TA,0.17,99.83
Undefined,0.00,100.00


Missing agent bookings:
16340

Missing company bookings:
112593


In [174]:

# ==========================================
# STEP 2C: INVESTIGATE MISSING CHILDREN
# ==========================================

# Extract records where children is missing
missing_children = df[df["children"].isnull()]

print("Number of missing children records:")
print(len(missing_children))


# Display relevant booking information
display(
    missing_children[
        [
            "hotel",
            "adults",
            "children",
            "babies",
            "market_segment",
            "customer_type",
            "is_canceled"
        ]
    ]
)


Number of missing children records:
4


,hotel,adults,children,babies,market_segment,customer_type,is_canceled
40600,City Hotel,2,NaN,0,Undefined,Transient-Party,1
40667,City Hotel,2,NaN,0,Direct,Transient-Party,1
40679,City Hotel,3,NaN,0,Undefined,Transient-Party,1
41160,City Hotel,2,NaN,0,Online TA,Transient-Party,1


In [175]:

# ==========================================
# STEP 2D: INVESTIGATE MISSING COUNTRY
# ==========================================

# Separate records with and without country information
df["country_missing"] = df["country"].isnull()


# Compare cancellation behaviour
country_analysis = (
    df.groupby("country_missing")
      .agg(
          bookings=("is_canceled", "count"),
          cancellation_rate=("is_canceled", "mean")
      )
)

country_analysis["cancellation_rate"] *= 100

display(country_analysis.round(2))


# Check missing country records by hotel
display(
    pd.crosstab(
        df["hotel"],
        df["country_missing"]
    )
)


,bookings,cancellation_rate
country_missing,,
False,118902,37.14
True,488,13.73


country_missing,False,True
hotel,,
City Hotel,79306,24
Resort Hotel,39596,464


In [176]:
pd.set_option('display.max_columns', None)

In [177]:

# ==========================================
# STEP 3A: DUPLICATE RECORD INVESTIGATION
# ==========================================

# Count duplicate records, excluding the first occurrence
duplicate_count = df.duplicated().sum()

# Identify all rows belonging to duplicate groups
duplicate_rows = df[df.duplicated(keep=False)]

# Calculate percentage of rows involved in duplicate groups
duplicate_percentage = (
    len(duplicate_rows) / len(df)
) * 100

print("========== DUPLICATE ANALYSIS ==========")

print("Total bookings:", len(df))

print("Duplicate records (excluding first occurrence):",
      duplicate_count)

print("Rows involved in duplicate groups:",
      len(duplicate_rows))

print(f"Percentage of rows involved in duplicate groups: "
      f"{duplicate_percentage:.2f}%")


# ==========================================
# STEP 3B: DUPLICATES BY HOTEL
# ==========================================

print("\n========== DUPLICATES BY HOTEL ==========")

duplicate_by_hotel = (
    duplicate_rows.groupby("hotel")
    .size()
    .reset_index(name="duplicate_group_rows")
)

display(duplicate_by_hotel)


# ==========================================
# STEP 3C: INSPECT SAMPLE DUPLICATES
# ==========================================

print("\n========== SAMPLE DUPLICATE RECORDS ==========")

# Display a few identical records
display(
    duplicate_rows.sort_values(
        by=list(df.columns)
    ).head(10)
)


========== DUPLICATE ANALYSIS ==========
Total bookings: 119390
Duplicate records (excluding first occurrence): 31994
Rows involved in duplicate groups: 40165
Percentage of rows involved in duplicate groups: 33.64%

========== DUPLICATES BY HOTEL ==========


,hotel,duplicate_group_rows
0,City Hotel,31748
1,Resort Hotel,8417



========== SAMPLE DUPLICATE RECORDS ==========


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,agent_missing,company_missing,country_missing
40772,City Hotel,0,0,2015,August,32,7,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09,2015-08-07,False,True,False
40802,City Hotel,0,0,2015,August,32,7,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09,2015-08-07,False,True,False
40821,City Hotel,0,0,2015,August,32,8,0,1,2,0.0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09,2015-08-08,False,True,False
40838,City Hotel,0,0,2015,August,32,8,0,1,2,0.0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09,2015-08-08,False,True,False
76792,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11,2015-08-10,True,True,False
76793,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11,2015-08-10,True,True,False
76794,City Hotel,0,0,2015,August,33,10,1,0,2,0.0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11,2015-08-10,True,True,False
41067,City Hotel,0,0,2015,August,33,11,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12,2015-08-11,True,False,False
41073,City Hotel,0,0,2015,August,33,11,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12,2015-08-11,True,False,False
41071,City Hotel,0,0,2015,August,33,11,0,1,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,NaN,NaN,0,Transient,80.0,0,0,Check-Out,2015-08-12,2015-08-11,True,True,False


In [178]:

# Count only repeated records, excluding the first occurrence

duplicate_records = df[df.duplicated(keep="first")]

# Calculate duplicate counts by hotel
duplicate_by_hotel = (
    duplicate_records.groupby("hotel")
    .size()
    .reset_index(name="duplicate_records")
)

display(duplicate_by_hotel)


,hotel,duplicate_records
0,City Hotel,25902
1,Resort Hotel,6092


In [179]:

# ==========================================
# STEP 3E: IMPACT OF DUPLICATE REMOVAL
# ==========================================

# Get records identified as duplicates
duplicate_records = df[df.duplicated(keep="first")]

# Select one duplicate record
sample_record = duplicate_records.iloc[0]

# Identify its hotel and arrival date
selected_hotel = sample_record["hotel"]
selected_date = sample_record["arrival_date"]

print("Selected Hotel:", selected_hotel)
print("Selected Arrival Date:", selected_date)


# Count bookings for this hotel-night BEFORE removal
bookings_before = df[
    (df["hotel"] == selected_hotel) &
    (df["arrival_date"] == selected_date)
]

# Create a separate dataset without exact duplicates
df_without_duplicates = df.drop_duplicates()

# Count bookings for the same hotel-night AFTER removal
bookings_after = df_without_duplicates[
    (df_without_duplicates["hotel"] == selected_hotel) &
    (df_without_duplicates["arrival_date"] == selected_date)
]


# Compare the results
print("\n========== DUPLICATE IMPACT ==========")

print("Bookings before removal:", len(bookings_before))

print("Bookings after removal:", len(bookings_after))

print("Records removed:", len(bookings_before) - len(bookings_after))

print(
    "Percentage reduction:",
    round(
        (len(bookings_before) - len(bookings_after))
        / len(bookings_before) * 100,
        2
    ),
    "%"
)


# Compare actual cancellations as well
print("\nCancellations before removal:",
      bookings_before["is_canceled"].sum())

print("Cancellations after removal:",
      bookings_after["is_canceled"].sum())


Selected Hotel: Resort Hotel
Selected Arrival Date: 2015-07-01 00:00:00

========== DUPLICATE IMPACT ==========
Bookings before removal: 43
Bookings after removal: 41
Records removed: 2
Percentage reduction: 4.65 %

Cancellations before removal: 5
Cancellations after removal: 5


In [180]:

# Investigate unusual ADR values

print("========== ADR ANOMALY ANALYSIS ==========")

print("\nADR Summary:")
print(df["adr"].describe())

print("\nNegative ADR records:")
negative_adr = df[df["adr"] < 0]
print("Count:", len(negative_adr))
display(
    negative_adr[
        ["hotel", "adr", "adults", "children", "babies",
         "stays_in_weekend_nights", "stays_in_week_nights",
         "is_canceled", "market_segment"]
    ].head(10)
)

print("\nZero ADR records:")
zero_adr = df[df["adr"] == 0]
print("Count:", len(zero_adr))

print("\nVery high ADR records (ADR > 1000):")
high_adr = df[df["adr"] > 1000]
print("Count:", len(high_adr))
display(
    high_adr[
        ["hotel", "adr", "adults", "children", "babies",
         "stays_in_weekend_nights", "stays_in_week_nights",
         "is_canceled", "market_segment"]
    ].head(10)
)


========== ADR ANOMALY ANALYSIS ==========

ADR Summary:
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

Negative ADR records:
Count: 1


,hotel,adr,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,is_canceled,market_segment
14969,Resort Hotel,-6.38,2,0.0,0,4,6,0,Groups



Zero ADR records:
Count: 1959

Very high ADR records (ADR > 1000):
Count: 1


,hotel,adr,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,is_canceled,market_segment
48515,City Hotel,5400.0,2,0.0,0,0,1,1,Offline TA/TO


In [181]:

print("========== STAY AND GUEST ANOMALIES ==========")

# Derive nights here so this cell works when run independently.
df["nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# 1. Zero-night bookings
zero_nights = df[df["nights"] == 0]

print("\nZero-night bookings:")
print("Count:", len(zero_nights))

display(
    zero_nights[
        ["hotel", "adults", "children", "babies",
         "adr", "is_canceled", "market_segment",
         "customer_type"]
    ].head(10)
)

# 2. Zero-guest bookings
zero_guests = df[
    (df["adults"] == 0) &
    (df["children"] == 0) &
    (df["babies"] == 0)
]

print("\nZero-guest bookings:")
print("Count:", len(zero_guests))

display(
    zero_guests[
        ["hotel", "adults", "children", "babies",
         "adr", "nights", "is_canceled",
         "market_segment"]
    ].head(10)
)

# 3. Negative ADR impact on empty-room value
negative_adr = df[df["adr"] < 0].copy()
negative_adr["empty_room_value"] = (
    negative_adr["adr"] * negative_adr["nights"]
)

print("\nNegative ADR empty-room value:")
display(
    negative_adr[
        ["hotel", "adr", "nights", "empty_room_value"]
    ]
)

========== STAY AND GUEST ANOMALIES ==========

Zero-night bookings:
Count: 715


,hotel,adults,children,babies,adr,is_canceled,market_segment,customer_type
0,Resort Hotel,2,0.0,0,0.0,0,Direct,Transient
1,Resort Hotel,2,0.0,0,0.0,0,Direct,Transient
167,Resort Hotel,2,0.0,0,0.0,0,Online TA,Transient
168,Resort Hotel,1,0.0,0,0.0,0,Direct,Transient
196,Resort Hotel,2,0.0,0,0.0,0,Direct,Transient
197,Resort Hotel,2,0.0,0,0.0,0,Online TA,Transient
459,Resort Hotel,2,0.0,0,0.0,0,Online TA,Transient
568,Resort Hotel,1,0.0,0,0.0,0,Offline TA/TO,Transient
569,Resort Hotel,2,0.0,0,0.0,0,Online TA,Transient
618,Resort Hotel,2,2.0,0,0.0,0,Online TA,Transient



Zero-guest bookings:
Count: 180


,hotel,adults,children,babies,adr,nights,is_canceled,market_segment
2224,Resort Hotel,0,0.0,0,0.0,3,0,Corporate
2409,Resort Hotel,0,0.0,0,0.0,0,0,Corporate
3181,Resort Hotel,0,0.0,0,0.0,3,0,Groups
3684,Resort Hotel,0,0.0,0,0.0,5,0,Groups
3708,Resort Hotel,0,0.0,0,0.0,6,0,Groups
4127,Resort Hotel,0,0.0,0,0.0,0,1,Offline TA/TO
9376,Resort Hotel,0,0.0,0,0.0,0,1,Offline TA/TO
31765,Resort Hotel,0,0.0,0,28.0,10,0,Direct
32029,Resort Hotel,0,0.0,0,0.0,1,0,Groups
32827,Resort Hotel,0,0.0,0,0.0,12,0,Direct



Negative ADR empty-room value:


,hotel,adr,nights,empty_room_value
14969,Resort Hotel,-6.38,10,-63.8


In [182]:

# Hotel-night level analysis

hotel_night = (
    df.groupby(["hotel", "arrival_date"])
      .agg(
          total_bookings=("is_canceled", "count"),
          cancellations=("is_canceled", "sum"),
          arrivals=("is_canceled", lambda x: (x == 0).sum()),
          cancellation_rate=("is_canceled", "mean")
      )
      .reset_index()
)

hotel_night["cancellation_rate"] *= 100

print("========== HOTEL-NIGHT ANALYSIS ==========")

print("\nTotal hotel-nights:", len(hotel_night))

print("\nHotel-night count by hotel:")
display(
    hotel_night.groupby("hotel")
    .agg(
        hotel_nights=("arrival_date", "count"),
        average_bookings=("total_bookings", "mean"),
        median_bookings=("total_bookings", "median"),
        maximum_bookings=("total_bookings", "max"),
        average_cancellation_rate=("cancellation_rate", "mean")
    ).round(2)
)

print("\nHotel-night booking distribution:")
display(
    hotel_night.groupby("hotel")["total_bookings"]
    .describe()
    .round(2)
)

print("\nSample hotel-night records:")
display(hotel_night.head(10))


========== HOTEL-NIGHT ANALYSIS ==========

Total hotel-nights: 1586

Hotel-night count by hotel:


,hotel_nights,average_bookings,median_bookings,maximum_bookings,average_cancellation_rate
hotel,,,,,
City Hotel,793,100.04,97.0,340,37.91
Resort Hotel,793,50.52,46.0,222,25.24



Hotel-night booking distribution:


,count,mean,std,min,25%,50%,75%,max
hotel,,,,,,,,
City Hotel,793.0,100.04,52.77,6.0,64.0,97.0,128.0,340.0
Resort Hotel,793.0,50.52,23.61,5.0,35.0,46.0,62.0,222.0



Sample hotel-night records:


,hotel,arrival_date,total_bookings,cancellations,arrivals,cancellation_rate
0,City Hotel,2015-07-01,79,14,65,17.721519
1,City Hotel,2015-07-02,49,48,1,97.959184
2,City Hotel,2015-07-03,16,6,10,37.500000
3,City Hotel,2015-07-04,38,29,9,76.315789
4,City Hotel,2015-07-05,8,8,0,100.000000
5,City Hotel,2015-07-06,30,29,1,96.666667
6,City Hotel,2015-07-07,19,17,2,89.473684
7,City Hotel,2015-07-08,40,16,24,40.000000
8,City Hotel,2015-07-09,46,45,1,97.826087
9,City Hotel,2015-07-10,9,7,2,77.777778


In [183]:

# Validate the official project periods

train_df, choose_df, proof_df = contract.split_periods(df)

print("========== PROJECT PERIOD VALIDATION ==========")

print("\nDataset shapes:")
print("Full dataset:", df.shape)
print("Train:", train_df.shape)
print("Choose:", choose_df.shape)
print("Proof:", proof_df.shape)

print("\nTotal rows across periods:")
print(len(train_df) + len(choose_df) + len(proof_df))

print("\nArrival date ranges:")
for name, period_df in [
    ("Train", train_df),
    ("Choose", choose_df),
    ("Proof", proof_df)
]:
    print(
        f"{name}: "
        f"{period_df['arrival_date'].min()} to "
        f"{period_df['arrival_date'].max()}"
    )

print("\nHotel-night counts:")
for name, period_df in [
    ("Train", train_df),
    ("Choose", choose_df),
    ("Proof", proof_df)
]:
    print(
        name,
        period_df.groupby(["hotel", "arrival_date"])
                 .ngroups
    )


========== PROJECT PERIOD VALIDATION ==========

Dataset shapes:
Full dataset: (119390, 37)
Train: (49157, 37)
Choose: (29546, 37)
Proof: (40687, 37)

Total rows across periods:
119390

Arrival date ranges:
Train: 2015-07-01 00:00:00 to 2016-06-30 00:00:00
Choose: 2016-07-01 00:00:00 to 2016-12-31 00:00:00
Proof: 2017-01-01 00:00:00 to 2017-08-31 00:00:00

Hotel-night counts:
Train 732
Choose 368
Proof 486


In [184]:

# Inspect the project contract module

import inspect

print("========== CONTRACT MODULE ==========")

print(inspect.getsource(contract))


========== CONTRACT MODULE ==========
"""The client's scoring rule, as code. Read it; do not change it.

The hotel group judges any oversell policy the same way, whichever method produced it, so every team
is scored on the same terms. This file holds that rule and nothing else: it does not choose your
bookings, your columns, your method or your buffer. Those are your team's decisions.

The rule in words (docs/EVALUATION_CONTRACT.md says the same thing):
  - The unit is one hotel on one arrival night. A night's book is every booking due to arrive that night.
  - Your policy sells k extra rooms for that night. If more bookings cancel than k, the difference
    stays empty; if fewer cancel than k, the difference is guests walked to another hotel.
  - An empty room costs the night's average booked stay value (ADR x nights of that night's bookings).
  - A walked guest costs WALK_COST euros.
  - A policy is acceptable only if it walks at most WALK_CEILING guests per 1,000 arrivals.
  - Train

In [185]:

from pathlib import Path

project_root = Path.cwd().parent

print("========== PROJECT STRUCTURE ==========")
print("Project root:", project_root)

for folder in ["src", "tests", "docs", "data"]:
    folder_path = project_root / folder

    print(f"\n{folder}/")

    if folder_path.exists():
        for file in sorted(folder_path.iterdir()):
            if file.is_file():
                print("  ", file.name)
            elif file.is_dir():
                print("   ", file.name, "/")
    else:
        print("   Folder not found")


========== PROJECT STRUCTURE ==========
Project root: d:\IITGN\Group_project_3_Hotel_booking\C2_W06_Build2_Student_Pack\build2-starter

src/
    __pycache__ /
   contract.py
   data.py
   features.py
   README.md

tests/
    __pycache__ /
   test_contract.py

docs/
   AI_WORKFLOW.md
   BUILD_PLAN.md
   CHARTER.md
   COLUMNS.md
   COMPARISON.md
   DECISIONS.md
   DOMAIN.md
   EVALUATION_CONTRACT.md
   grill_01_charter.md
   grill_02_prd.md
    img /
   LADDER.md
   PRD.md
   PRESENTATION.md
   RECORDING.md
   RFP.md
   ROADMAP.md
   ROLES.md
   RUBRIC.md

data/
   hotels.csv
   README.md


In [186]:

from pathlib import Path

project_root = Path.cwd().parent

files_to_read = [
    "docs/COLUMNS.md",
    "docs/DECISIONS.md",
    "src/README.md"
]

for file_name in files_to_read:
    file_path = project_root / file_name

    print("\n" + "=" * 70)
    print(f"FILE: {file_name}")
    print("=" * 70)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File not found.")



FILE: docs/COLUMNS.md
# Which columns did the hotel know on the day of booking?

**Owner:** Data Lead. **Closes:** part of ladder stage 1. **Used by:** `src/features.py`, the PRD and slide 4.

Put every column in exactly one bin, by when the hotel learns its value:

1. **Fixed at booking**: known on the day the booking is made, and it does not change.
2. **Amendable**: known at booking but it can change before arrival, or it is only partly known at booking.
3. **After the fact**: written at check-in, during the stay, or when the outcome is already known.

The project rubric requires the chosen rule to use only the first bin. A second-bin column may be tried as a caveated experiment; a third-bin column must not feed a booking-time method.

**Important dataset limitation:** Antonio et al. (2019) explain that, to avoid target leakage, values were extracted from the booking change log using a timestamp relative to the day before arrival (for bookings created before arrival). Therefore, th

In [196]:
# Validate the source-column timing audit and the team-approved features

# Bin definitions match the audited register in docs/COLUMNS.md.
bin_1 = [
    "hotel", "market_segment", "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled", "agent",
    "company", "customer_type",
]
bin_2 = [
    "lead_time", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults",
    "children", "babies", "meal", "reserved_room_type", "deposit_type",
    "days_in_waiting_list", "required_car_parking_spaces",
    "total_of_special_requests",
]
bin_3 = [
    "is_canceled", "country", "assigned_room_type", "booking_changes",
    "adr", "reservation_status", "reservation_status_date",
]

original_columns = list(data.read_raw().columns)
classified_columns = bin_1 + bin_2 + bin_3
selected = features.MODEL_FEATURES
selected_by_bin = {
    "Bin 1": [c for c in selected if c in bin_1],
    "Bin 2": [c for c in selected if c in bin_2],
    "Bin 3": [c for c in selected if c in bin_3],
}

print("Original columns:", len(original_columns))
print("Timing bins: Bin 1 =", len(bin_1), "| Bin 2 =", len(bin_2), "| Bin 3 =", len(bin_3))
print("Classified columns:", len(classified_columns))
print("Unique classified columns:", len(set(classified_columns)))
print("Unclassified:", sorted(set(original_columns) - set(classified_columns)))
print("Not in source:", sorted(set(classified_columns) - set(original_columns)))
print("Duplicates across bins:", sorted(c for c in set(classified_columns) if classified_columns.count(c) > 1))
print("Team-approved feature count:", len(selected))
print("Selected features by bin:", {k: len(v) for k, v in selected_by_bin.items()})
print("Selected timing-sensitive fields:", selected_by_bin["Bin 2"] + selected_by_bin["Bin 3"])


Original columns: 32
Timing bins: Bin 1 = 9 | Bin 2 = 16 | Bin 3 = 7
Classified columns: 32
Unique classified columns: 32
Unclassified: []
Not in source: []
Duplicates across bins: []
Team-approved feature count: 17
Selected features by bin: {'Bin 1': 9, 'Bin 2': 7, 'Bin 3': 1}
Selected timing-sensitive fields: ['lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'meal', 'reserved_room_type', 'country']


In [188]:

import inspect

print("========== DATA MODULE ==========")
print(inspect.getsource(data))


========== DATA MODULE ==========
"""Get the client's booking history and hand it over exactly as the hotels' system exported it.

The bookings are the public Hotel Booking Demand datasets: Antonio, N., de Almeida, A. and Nunes, L.
(2019). Hotel booking demand datasets. Data in Brief, 22, 41-49. https://doi.org/10.1016/j.dib.2018.11.126
The article and its data are open access under CC BY 4.0, so every copy of your work cites it.
The copy downloaded here is the cleaned-column version published by the TidyTuesday project
(checked 7 November 2026).
"""
import pathlib
import urllib.request

import pandas as pd

URL = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/"
       "data/2020/2020-02-11/hotels.csv")  # checked 7 November 2026
DATA_DIR = pathlib.Path(__file__).resolve().parent.parent / "data"
FILE = "hotels.csv"

# The outcome column: 1 if the booking was cancelled (including no-shows), 0 if the guest arrived.
TARGET = "is_canceled"

EXPECTED_ROWS = 119_390
E

In [189]:

from pathlib import Path

project_root = Path.cwd().parent

files_to_read = [
    "docs/BUILD_PLAN.md",
    "docs/CHARTER.md",
    "docs/PRD.md"
]

for file_name in files_to_read:
    file_path = project_root / file_name

    print("\n" + "=" * 75)
    print(f"FILE: {file_name}")
    print("=" * 75)

    print(file_path.read_text(encoding="utf-8"))



FILE: docs/BUILD_PLAN.md
# How do we get from a first test to something the client can run?

**Owner:** Build Lead. **Closes:** ladder stages 4 and 6. **Length:** one page.

Three stages, each with an owner, an hour budget, a line that says when it is done, and the file that proves it. Fill the table on Wednesday morning and update the hours as you go.

| Stage | What is built | Owner | Hours planned | Hours used | Done when | Evidence file |
|---|---|---|---|---|---|---|
| POC | | | | | Every method you tried prints from `python src/backtest.py` on the proof period, scored by `src/contract.py`, with the ceiling checked | |
| MVP | | | | | `python src/nightly.py --from 2017-08-18 --nights 14` writes the nightly table from a bookings file, reading only first-bin columns, signed by the challenger | |
| Final | | | | | The uncertainty buffer is compared with the rooms buffer, the full-night caveat is computed, and the roadmap is written | |

In the team of three, the final stage is the f

In [190]:

# Investigate timing-sensitive columns

timing_columns = [
    "adr",
    "deposit_type",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "agent",
    "company",
    "country",
    "assigned_room_type",
    "booking_changes"
]

for col in timing_columns:
    print("\n" + "=" * 55)
    print("COLUMN:", col)
    print("Missing values:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

    print("\nMost frequent values:")
    print(df[col].value_counts(dropna=False).head(10))



COLUMN: adr
Missing values: 0
Unique values: 8879

Most frequent values:
adr
62.0     3754
75.0     2715
90.0     2473
65.0     2418
0.0      1959
80.0     1889
95.0     1661
120.0    1607
100.0    1573
85.0     1538
Name: count, dtype: int64

COLUMN: deposit_type
Missing values: 0
Unique values: 3

Most frequent values:
deposit_type
No Deposit    104641
Non Refund     14587
Refundable       162
Name: count, dtype: int64

COLUMN: required_car_parking_spaces
Missing values: 0
Unique values: 5

Most frequent values:
required_car_parking_spaces
0    111974
1      7383
2        28
3         3
8         2
Name: count, dtype: int64

COLUMN: total_of_special_requests
Missing values: 0
Unique values: 6

Most frequent values:
total_of_special_requests
0    70318
1    33226
2    12969
3     2497
4      340
5       40
Name: count, dtype: int64

COLUMN: agent
Missing values: 16340
Unique values: 333

Most frequent values:
agent
9.0      31961
NaN      16340
240.0    13922
1.0       7191
14.0     

In [191]:

# Deposit type cancellation analysis

deposit_detail = (
    df.groupby(["hotel", "deposit_type"])
      .agg(
          bookings=("is_canceled", "count"),
          cancellations=("is_canceled", "sum"),
          cancellation_rate=("is_canceled", "mean")
      )
      .reset_index()
)

deposit_detail["cancellation_rate"] *= 100

print("========== DEPOSIT TYPE BY HOTEL ==========")
display(deposit_detail.round(2))

print("\n========== DEPOSIT TYPE BY MARKET SEGMENT ==========")

deposit_segment = (
    df.groupby(["market_segment", "deposit_type"])
      .agg(
          bookings=("is_canceled", "count"),
          cancellation_rate=("is_canceled", "mean")
      )
      .reset_index()
)

deposit_segment["cancellation_rate"] *= 100

display(deposit_segment.round(2))


========== DEPOSIT TYPE BY HOTEL ==========


,hotel,deposit_type,bookings,cancellations,cancellation_rate
0,City Hotel,No Deposit,66442,20244,30.47
1,City Hotel,Non Refund,12868,12844,99.81
2,City Hotel,Refundable,20,14,70.00
3,Resort Hotel,No Deposit,38199,9450,24.74
4,Resort Hotel,Non Refund,1719,1650,95.99
5,Resort Hotel,Refundable,142,22,15.49



========== DEPOSIT TYPE BY MARKET SEGMENT ==========


,market_segment,deposit_type,bookings,cancellation_rate
0,Aviation,No Deposit,237,21.94
1,Complementary,No Deposit,743,13.06
2,Corporate,No Deposit,4957,13.70
3,Corporate,Non Refund,334,93.41
4,Corporate,Refundable,4,25.00
5,Direct,No Deposit,12581,15.23
6,Direct,Non Refund,19,84.21
7,Direct,Refundable,6,33.33
8,Groups,No Deposit,10509,28.30
9,Groups,Non Refund,9172,99.32


In [192]:

# Country missingness and cancellation analysis

country_analysis = (
    df.assign(country_status=df["country"].isna().map({
        True: "Missing",
        False: "Available"
    }))
    .groupby(["hotel", "country_status"])
    .agg(
        bookings=("is_canceled", "count"),
        cancellations=("is_canceled", "sum"),
        cancellation_rate=("is_canceled", "mean")
    )
    .reset_index()
)

country_analysis["cancellation_rate"] *= 100

display(country_analysis.round(2))


,hotel,country_status,bookings,cancellations,cancellation_rate
0,City Hotel,Available,79306,33080,41.71
1,City Hotel,Missing,24,22,91.67
2,Resort Hotel,Available,39596,11077,27.98
3,Resort Hotel,Missing,464,45,9.70


In [193]:

for col in ["agent", "company"]:

    print(f"\n========== {col.upper()} ANALYSIS ==========")

    df[f"{col}_status"] = df[col].isna().map({
        True: "Missing",
        False: "Available"
    })

    result = (
        df.groupby(["hotel", "market_segment", f"{col}_status"])
          .agg(
              bookings=("is_canceled", "count"),
              cancellation_rate=("is_canceled", "mean")
          )
          .reset_index()
    )

    result["cancellation_rate"] *= 100

    display(result.round(2))



========== AGENT ANALYSIS ==========


,hotel,market_segment,agent_status,bookings,cancellation_rate
0,City Hotel,Aviation,Available,25,24.00
1,City Hotel,Aviation,Missing,212,21.70
2,City Hotel,Complementary,Available,53,11.32
3,City Hotel,Complementary,Missing,489,11.86
4,City Hotel,Corporate,Available,311,18.33
5,City Hotel,Corporate,Missing,2675,21.83
6,City Hotel,Direct,Available,3657,17.83
7,City Hotel,Direct,Missing,2436,16.58
8,City Hotel,Groups,Available,12073,68.79
9,City Hotel,Groups,Missing,1902,69.30



========== COMPANY ANALYSIS ==========


,hotel,market_segment,company_status,bookings,cancellation_rate
0,City Hotel,Aviation,Available,212,21.70
1,City Hotel,Aviation,Missing,25,24.00
2,City Hotel,Complementary,Available,289,13.15
3,City Hotel,Complementary,Missing,253,10.28
4,City Hotel,Corporate,Available,2595,22.31
5,City Hotel,Corporate,Missing,391,15.86
6,City Hotel,Direct,Available,60,6.67
7,City Hotel,Direct,Missing,6033,17.44
8,City Hotel,Groups,Available,402,18.41
9,City Hotel,Groups,Missing,13573,70.35


In [194]:

import importlib
import features

importlib.reload(features)

# Load a fresh copy of the original CSV.
raw_df = data.read_raw()

# Take a snapshot before preparation.
raw_snapshot = raw_df.copy(deep=True)

# Prepare the data.
prepared_df = features.prepare_bookings(raw_df)

print("========== CORRECTED VALIDATION ==========")

print("Raw dataset shape:", raw_df.shape)
print("Prepared dataset shape:", prepared_df.shape)

# Compare original data against its snapshot.
print("\nOriginal DataFrame unchanged:", raw_df.equals(raw_snapshot))

# Verify row preservation.
print("Rows preserved:", len(raw_df) == len(prepared_df))

# Verify missing values.
print("\nMissing values after cleaning:")
print(
    prepared_df[
        ["children", "country", "agent", "company"]
    ].isnull().sum()
)

# Verify duplicates are retained.
print("\nDuplicate records retained:")
print(prepared_df.duplicated().sum())

# Verify scoring columns.
print(
    "\nScoring columns available:",
    all(col in prepared_df.columns
        for col in features.SCORING_COLUMNS)
)


========== CORRECTED VALIDATION ==========
Raw dataset shape: (119390, 32)
Prepared dataset shape: (118563, 34)

Original DataFrame unchanged: True
Rows preserved: False

Missing values after cleaning:
children    0
country     0
agent       0
company     0
dtype: int64

Duplicate records retained:
31926

Scoring columns available: True


In [195]:
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,agent_missing,company_missing,country_missing,nights,agent_status,company_status
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,True,True,False,0,Missing,Missing
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,True,True,False,0,Missing,Missing
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,True,True,False,1,Missing,Missing
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,False,True,False,1,Available,Missing
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03,2015-07-01,False,True,False,2,Available,Missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,0.0,0,BB,BEL,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06,2017-08-30,False,True,False,7,Available,Missing
119386,City Hotel,0,102,2017,August,35,31,2,5,3,0.0,0,BB,FRA,Online TA,TA/TO,0,0,0,E,E,0,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07,2017-08-31,False,True,False,7,Available,Missing
119387,City Hotel,0,34,2017,August,35,31,2,5,2,0.0,0,BB,DEU,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07,2017-08-31,False,True,False,7,Available,Missing
119388,City Hotel,0,109,2017,August,35,31,2,5,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07,2017-08-31,False,True,False,7,Available,Missing


In [149]:
import importlib
import features

importlib.reload(features)

# Check the approved feature list
print("Number of model features:", len(features.MODEL_FEATURES))

print("\nApproved model features:")
print(features.get_feature_columns(df))

print("\nTarget variable:")
print(features.TARGET)

print("\nScoring columns:")
print(features.SCORING_COLUMNS)


Number of model features: 17

Approved model features:
['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'agent', 'company', 'customer_type']

Target variable:
is_canceled

Scoring columns:
['hotel', 'arrival_date', 'is_canceled', 'adr', 'nights']


In [150]:

import importlib
import features

importlib.reload(features)

# Prepare the original dataset.
prepared_df = features.prepare_bookings(df)

print("========== PREPARATION VALIDATION ==========")

print("Original shape:", df.shape)
print("Prepared shape:", prepared_df.shape)

print("\nOriginal dataset unchanged:")
print(df.shape == (119390, 37))

print("\nMissing values after cleaning:")
print(
    prepared_df[
        ["children", "country", "agent", "company"]
    ].isnull().sum()
)

print("\nDerived columns:")
display(
    prepared_df[
        ["arrival_date", "nights"]
    ].head()
)

print("\nRemaining duplicate records:")
print(prepared_df.duplicated().sum())

print("\nScoring columns available:")
print(
    all(
        column in prepared_df.columns
        for column in features.SCORING_COLUMNS
    )
)

print("\nTotal rows preserved:")
print(len(prepared_df) == len(df))


========== PREPARATION VALIDATION ==========
Original shape: (119390, 39)
Prepared shape: (118563, 39)

Original dataset unchanged:
False

Missing values after cleaning:
children    0
country     0
agent       0
company     0
dtype: int64

Derived columns:


,arrival_date,nights
2,2015-07-01,1
3,2015-07-01,1
4,2015-07-01,2
5,2015-07-01,2
6,2015-07-01,2



Remaining duplicate records:
31926

Scoring columns available:
True

Total rows preserved:
False


In [151]:

import importlib
import features

importlib.reload(features)

# Start from the prepared dataset.
X, y, scoring = features.split_model_inputs(prepared_df)

print("========== MODEL INPUT SEPARATION ==========")

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("Scoring shape:", scoring.shape)

print("\nModel feature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget cancellation rate:")
print(round(y.mean() * 100, 2), "%")

print("\nScoring columns:")
print(scoring.columns.tolist())

print("\nIndex alignment:")
print("X and y:", X.index.equals(y.index))
print("X and scoring:", X.index.equals(scoring.index))

print("\nTarget excluded from X:")
print(features.TARGET not in X.columns)

print("\nADR excluded from X:")
print("adr" not in X.columns)

print("\nNights excluded from X:")
print("nights" not in X.columns)


========== MODEL INPUT SEPARATION ==========

X shape: (118563, 17)
y shape: (118563,)
Scoring shape: (118563, 5)

Model feature columns:
['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'agent', 'company', 'customer_type']

Target distribution:
is_canceled
0    74388
1    44175
Name: count, dtype: int64

Target cancellation rate:
37.26 %

Scoring columns:
['hotel', 'arrival_date', 'is_canceled', 'adr', 'nights']

Index alignment:
X and y: True
X and scoring: True

Target excluded from X:
True

ADR excluded from X:
True

Nights excluded from X:
True


In [152]:

# ============================================================
# STEP 4: CREATE TEMPORAL MODEL DATASETS
# ============================================================

# Use the official project split.
train_df, choose_df, proof_df = contract.split_periods(prepared_df)

# Separate model inputs, target and scoring data for each period.
X_train, y_train, score_train = features.split_model_inputs(train_df)

X_choose, y_choose, score_choose = features.split_model_inputs(choose_df)

X_proof, y_proof, score_proof = features.split_model_inputs(proof_df)


print("========== TEMPORAL MODEL DATASETS ==========")

print("\nTRAIN")
print("Features:", X_train.shape)
print("Target:", y_train.shape)
print("Scoring:", score_train.shape)

print("\nCHOOSE")
print("Features:", X_choose.shape)
print("Target:", y_choose.shape)
print("Scoring:", score_choose.shape)

print("\nPROOF")
print("Features:", X_proof.shape)
print("Target:", y_proof.shape)
print("Scoring:", score_proof.shape)

print("\nCancellation rates:")
print("Train:", round(y_train.mean() * 100, 2), "%")
print("Choose:", round(y_choose.mean() * 100, 2), "%")
print("Proof:", round(y_proof.mean() * 100, 2), "%")

print("\nRow count validation:")
print(
    len(X_train) + len(X_choose) + len(X_proof)
    == len(prepared_df)
)

print("\nIndex alignment:")
print("Train:", X_train.index.equals(y_train.index))
print("Choose:", X_choose.index.equals(y_choose.index))
print("Proof:", X_proof.index.equals(y_proof.index))


========== TEMPORAL MODEL DATASETS ==========

TRAIN
Features: (48747, 17)
Target: (48747,)
Scoring: (48747, 5)

CHOOSE
Features: (29307, 17)
Target: (29307,)
Scoring: (29307, 5)

PROOF
Features: (40509, 17)
Target: (40509,)
Scoring: (40509, 5)

Cancellation rates:
Train: 36.03 %
Choose: 37.13 %
Proof: 38.83 %

Row count validation:
True

Index alignment:
Train: True
Choose: True
Proof: True
